# 🔄 Step 2: Feature Engineering & Preprocessing

Calculate spectral indices and prepare multi-band composite for classification.

## What This Notebook Does:
- ✅ Load raw satellite bands (B02, B03, B04, B08)
- ✅ Calculate **NDVI** (vegetation index)
- ✅ Calculate **NDWI** (water index)
- ✅ Calculate **Texture** (spatial variance)
- ✅ Optional: Clip to Area of Interest (AOI) using GeoJSON
- ✅ Stack all 7 features into single GeoTIFF

## Input Required:
- `../data/raw/coastalImage/` - Raw Sentinel-2 bands
- `../data/aoi/*.geojson` - (Optional) AOI boundary for clipping

## Output:
- `../data/processed/processed_image_with_indices.tif` - 7-band stacked image
  - Band 1: Blue (B02)
  - Band 2: Green (B03)
  - Band 3: Red (B04)
  - Band 4: NIR (B08)
  - Band 5: NDVI (vegetation)
  - Band 6: NDWI (water)
  - Band 7: Texture (spatial pattern)

---

**Previous:** [01_data_loading.ipynb](01_data_loading.ipynb)  
**Next:** [03_model_training.ipynb](03_model_training.ipynb)

## Imports

In [1]:
import rasterio
import numpy as np
import os
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter

## Clipping Function (Optional)

This function clips satellite imagery to a specific Area of Interest (AOI) defined by a GeoJSON file. This reduces file size and focuses analysis on the target region. 

In [2]:
import geopandas as gpd
from rasterio.mask import mask

def clip_to_aoi(band_path, geojson_path, output_path):
    # Load the GeoJSON AOI
    aoi = gpd.read_file(geojson_path)
    
    with rasterio.open(band_path) as src:
        # Ensure the GeoJSON and Image use the same projection
        aoi = aoi.to_crs(src.crs)
        
        # Mask the image using the AOI geometry
        out_image, out_transform = mask(src, aoi.geometry, crop=True)
        out_meta = src.meta.copy()

    # Update metadata for the new clipped file
    out_meta.update({
        "driver": "GTiff",
        "height": out_image.shape[1],
        "width": out_image.shape[2],
        "transform": out_transform
    })

    with rasterio.open(output_path, "w", **out_meta) as dest:
        dest.write(out_image)

## Multi-Index Calculation Function

This function computes three spectral indices from the 4 input bands:

**1. NDVI (Vegetation Index):**  
`(NIR - Red) / (NIR + Red)`
- Values > 0.3: Dense vegetation (seagrass, land forests)
- Values 0 to 0.3: Sparse vegetation, sand

- Values < 0: Water bodiesAll three indices are combined with the original 4 bands to create a 7-band composite.



**2. NDWI (Water Index):**  - Low texture: Smooth surfaces (open water, sand)

`(Green - NIR) / (Green + NIR)`- High texture: Rough surfaces (seaweed, coral)

- Positive values: Water bodiesLocal standard deviation in 3×3 window using NIR band

- Negative values: Land features**3. Texture (Spatial Variance):**  

- Helps distinguish water from wet sand

In [3]:
def load_and_prep_separate_tiffs(data_folder):
    # This dictionary maps the band name to the ID found in the filename
    band_keys = {
        'blue':  'B02', 
        'green': 'B03',
        'red':   'B04',
        'nir':   'B08'
    }
    
    bands = {}
    profile = None
    
    # Get a list of all files in the folder
    all_files = os.listdir(data_folder)

    print(f"Reading bands from {data_folder}...")
    for name, band_id in band_keys.items():
        # Search for a file that contains 'B02', 'B03', etc.
        target_file = None
        for f in all_files:
            if band_id in f and (f.endswith('.tif') or f.endswith('.tiff') or f.endswith('.jp2')):
                target_file = f
                break
        
        if target_file is None:
            print(f"❌ Error: Could not find a file containing {band_id} in {data_folder}")
            return None, None
            
        path = os.path.join(data_folder, target_file)
        with rasterio.open(path) as src:
            print(f"   ✅ Found {name}: {target_file}")
            bands[name] = src.read(1).astype('float32')
            if profile is None:
                profile = src.profile

    print("Calculating Smart Features...")
    np.seterr(divide='ignore', invalid='ignore')
    
    # 1. NDVI (Vegetation Index)
    ndvi = (bands['nir'] - bands['red']) / (bands['nir'] + bands['red'])
    
    # 2. NDWI (Water Index)
    ndwi = (bands['green'] - bands['nir']) / (bands['green'] + bands['nir'])

    # 3. TEXTURE (Variance/Roughness) - Using scipy (Fast & Robust)
    print("Calculating Texture...")
    img = bands['nir']
    mean = uniform_filter(img, size=3)
    sqr_mean = uniform_filter(img**2, size=3)
    var = sqr_mean - mean**2
    texture = np.sqrt(np.maximum(var, 0))

    # Clean up NaNs (Errors)
    ndvi = np.nan_to_num(ndvi, nan=0)
    ndwi = np.nan_to_num(ndwi, nan=0)
    texture = np.nan_to_num(texture, nan=0)

    # Stack all 7 layers
    stacked_image = np.stack([
        bands['blue'], bands['green'], bands['red'], bands['nir'], 
        ndvi, ndwi, texture
    ])
    
    # Update metadata
    profile.update(count=7, dtype='float32', driver='GTiff')

    return stacked_image, profile

## Execution


In [ ]:
# 1. Define paths
raw_folder = "../data/raw/coastalImage"
clipped_folder = "../data/processed/clipped_bands"

# Auto-detect any .geojson file in the data/aoi directory
aoi_dir = "../data/aoi"
if os.path.exists(aoi_dir):
    geojson_files = [os.path.join(aoi_dir, f) for f in os.listdir(aoi_dir) if f.endswith(".geojson")]
else:
    geojson_files = []
geojson_path = geojson_files[0] if geojson_files else None

# Check if we have a GeoJSON file for clipping
has_geojson = geojson_path is not None

if has_geojson:
    print(f"✅ Found GeoJSON file: {geojson_path}")
else:
    print("⚠️  No .geojson file found. Loading directly from raw folder...")

if has_geojson and not os.path.exists(clipped_folder):
    os.makedirs(clipped_folder)
    
    # 2. Clip each band before stacking (only if GeoJSON exists)
    print("✂️ Clipping bands to AOI...")
    for filename in os.listdir(raw_folder):
        if filename.endswith(".jp2") or filename.endswith(".tif") or filename.endswith(".tiff"):
            clip_to_aoi(
                os.path.join(raw_folder, filename),
                geojson_path,
                os.path.join(clipped_folder, filename)
            )
    data_folder = clipped_folder
    print("✅ Clipping complete!")
else:
    data_folder = raw_folder

# 3. Now run the stacking logic
print("🚀 Stacking bands...")
    output_path = "../data/processed/processed_image_with_indices.tif"
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(stacked_image)
    print(f"✅ Success! Processed image saved to {output_path}")
else:

    print("❌ Failed to process image. Check that your bands (B02, B03, B04, B08) exist in the folder.")
        dst.write(stacked_image)    print("❌ Failed to process image. Check that your bands (B02, B03, B04, B08) exist in the folder.")

    print("✅ Success! Processed image saved.")else:

✅ Found GeoJSON file: explorer-aoi.geojson
🚀 Stacking bands...
Reading bands from coastalImage...
   ✅ Found blue: B02.tiff
   ✅ Found green: B03.tiff
   ✅ Found red: B04.tiff
   ✅ Found nir: B08.tiff
Calculating Smart Features...
Calculating Texture...
✅ Success! Processed image saved.
